# 02 – Weather
Gets the 5-day / 3-hour forecast from OpenWeatherMap for every city in the database and appends it to the weather table.

## 1. Setup

In [51]:
# --- libraries ---
import os                          # read values from the .env file
import requests                    # send requests to the API
import pandas as pd                # DataFrames + read_sql / to_sql
from dotenv import load_dotenv     # reads the .env file that holds our secrets

load_dotenv()                                   # read the .env file in this folder
connection_string = os.getenv("CON_STRING")     # MySQL address incl. password
weather_key = os.getenv("WEATHER_KEY")          # OpenWeatherMap API key

# check WITHOUT printing the secrets themselves
print("connection string loaded:", connection_string is not None)
print("weather key loaded:", weather_key is not None)

connection string loaded: True
weather key loaded: True


## 2. Explore the API response (one city)

In [9]:
# one test call for Berlin: does the key work? (200 = success, 401 = key not active / wrong)
url = "https://api.openweathermap.org/data/2.5/forecast"                   # 5-day / 3-hour forecast endpoint
params = {"lat": 52.52, "lon": 13.405, "appid": weather_key, "units": "metric"}   # metric = °C and m/s

response = requests.get(url, params=params)
response.status_code

200

In [14]:
weather_json = response.json()     # JSON -> Python dictionary
weather_json.keys()                # 'list' holds the forecasts, 'city' holds location info

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])

In [16]:
len(weather_json["list"])          # 40 = 8 slots per day (every 3 hours) x 5 days

40

In [ ]:
weather_json["list"][0]            # one forecast slot: temperature in 'main', wind, 'pop' (rain probability) ...

## 3. Pipeline functions

In [56]:
def get_weather(cities_df):
    """Get the 5-day / 3-hour forecast for every city in cities_df.
    Returns 40 rows per city, linked by city_id."""
    url = "https://api.openweathermap.org/data/2.5/forecast"
    rows = []                                                  # one dictionary per forecast slot

    for _, city in cities_df.iterrows():                       # one city (row of the cities table) at a time
        params = {"lat":   city["latitude"],                   # coordinates come from the database
                  "lon":   city["longitude"],
                  "appid": weather_key,
                  "units": "metric"}                           # °C and m/s
        response = requests.get(url, params=params)

        if response.status_code != 200:                        # safety: skip a failed city
            print(f"⚠️ Skipped {city['city_name']}: status {response.status_code}")
            continue

        for forecast in response.json()["list"]:               # the 40 time slots
            rows.append({
                "city_id":       city["city_id"],                       # link to the cities table
                "forecast_time": forecast["dt_txt"],                    # start of the 3-hour slot (UTC)
                "outlook":       forecast["weather"][0]["description"], # 'weather' is a list -> [0]
                "temperature_c": forecast["main"]["temp"],              # cold weather = fewer riders
                "feels_like_c":  forecast["main"]["feels_like"],        # what people actually feel
                "wind_speed_ms": forecast["wind"]["speed"],             # strong wind = unpleasant / unsafe
                "rain_prob":     forecast["pop"],                       # probability of precipitation, 0-1
                # rain / snow keys only exist when precipitation is expected -> .get() with default 0
                "rain_3h_mm":    forecast.get("rain", {}).get("3h", 0),
                "snow_3h_mm":    forecast.get("snow", {}).get("3h", 0)
            })

    weather_df = pd.DataFrame(rows)
    weather_df["forecast_time"] = pd.to_datetime(weather_df["forecast_time"])   # text -> datetime
    weather_df["retrieved_at"] = pd.Timestamp.now().floor("s")                  # when we collected it
    return weather_df

In [58]:
def load_weather(weather_df):
    """Append the forecasts to the weather table."""
    weather_df.to_sql("weather",
                      if_exists="append",
                      con=connection_string,
                      index=False)
    print(f"✅ {len(weather_df)} weather rows added")


def run_weather_pipeline():
    """Read cities from MySQL, get their forecasts, store them."""
    cities_df = pd.read_sql("cities", con=connection_string)   # fresh read every run → new cities included
    print(f"🏙️ {len(cities_df)} cities found in the database")
    weather_df = get_weather(cities_df)
    load_weather(weather_df)

## 4. Dry run (does NOT write to MySQL)

In [ ]:
# dry run: get the forecasts without writing to MySQL
cities_df = pd.read_sql("cities", con=connection_string)   # cities from the database (source of truth)
weather_df = get_weather(cities_df)
weather_df["city_id"].value_counts()                        # should be 40 per city

## 5. Run the pipeline (writes to MySQL)
⚠️ Each run appends 40 rows per city with a new `retrieved_at`.

In [59]:
# Remove the # to run: appends a new forecast snapshot for all cities in the database.
# run_weather_pipeline()

🏙️ 4 cities found in the database
✅ 160 weather rows added
